In [12]:
from pydantic import BaseModel, EmailStr, AnyUrl, Field, field_validator,model_validator,computed_field
from typing import List, Dict, Optional, Annotated

In [13]:
#Model_validator
#Combine two field and perform validation, eg: (Age + contact details ) if age 
#greater than 60 than emergency contact details should be added. 

In [14]:
pip install pydantic

Note: you may need to restart the kernel to use updated packages.


In [15]:
#!pip install 'pydantic[email]'
import sys
!{sys.executable} -m pip install email-validator

In [24]:
# 1. Define the model
class Patient(BaseModel):
    name: Annotated[str, Field(max_length=50, title = 'Name of the patient', 
                               description='Give the name of the patient in less than 50 char',
                              examples=['Aditi', ['Reeva']])]  #str = Field(max_length=50)
    email: EmailStr
    linkedinUrl : AnyUrl
    age :int = Field(gt=0,lt=110)
    height : float #meters
    weight : float #kg
    #weight : Annotated[float, Field(gt=0,strict=True)]  #float= Field(gt=0)
                        #strict here is strictly accepting only float values 
    married : Annotated[bool, Field(default=None, description='Is the patient married or not')]   #bool = False
    allergies : Annotated[Optional[List[str]],Field(default=None,max_length=5)]  #Optional[List[str]] = Field(max_length=5)  #None
    contact_details : Dict[str,str]
        
        
    @field_validator('email')
    @classmethod  
    def email_validator(cls,value):
        valid_domains = ['hdfc.com','icici.com']
        #abc@gmail.com
        domain_name = value.split('@')[-1]
        if domain_name not in valid_domains:
            raise valueError('Not a valid domain')
        return value
    
    @field_validator('name')
    @classmethod  
    def transform_name(cls,value):
        #name should be in capital letter only
        return value.upper()
    
    @field_validator('age', mode='before')    #Default mode is 'After', type coecion se pehle wali value hogi
    @classmethod  
    def validate_age(cls,value):
        #age 
        if 0 < value < 100:   #passing the age value in string throws valueError in before mode while success in 'After' mode.
            return value
        else:
            raise valueError('Age should be in between 0 to 100')
            
    #if patient age is greater than 60 then emergency contact should be there in contact details.
    @model_validator(mode='after')
    def validate_emergency_contact(cls, model):
        if model.age > 60 and 'emergency' not in model.contact_details :
            raise ValueError('Patients older than 60 must have emergency contact')
        return model
            
        
    @computed_field
    @property
    def calculate_bmi(self)->float:
        bmi = round((self.weight)/(self.height**2),2)
        return bmi
        

/var/folders/7x/g3519v5d20q1_9ldldymv8b40000gn/T/ipykernel_16316/1943708148.py:44: PydanticDeprecatedSince212: Using `@model_validator` with mode='after' on a classmethod is deprecated. Instead, use an instance method. See the documentation at https://docs.pydantic.dev/2.12/concepts/validators/#model-after-validator. Deprecated in Pydantic V2.12 to be removed in V3.0.
  @model_validator(mode='after')


In [34]:
# 2. Define the function — it RECEIVES a patient, doesn't create one
def insert_patient_data(patient: Patient):
    print(patient.name)
    print(patient.email)
    print(patient.linkedinUrl)
    print(patient.age)
    print(patient.weight)
    print(patient.allergies)
    print(patient.married)
    print('BMI',patient.calculate_bmi)
    print("inserted")

In [35]:
# 3. Create patient1  
patient_info = {'name' : 'AditiGupta', 
                'email' : 'aditi.gup30@icici.com',
                'linkedinUrl' : 'http://www.linkedin.com',
                'age': 65,    #30, 
                'weight':75.3, 
                #'married' : True,
                'height' :1.72,
                'allergies':['pollen','dust','abc','xyz'],
                'contact_details':{'email':'aditi.gup30@gmail.com','phone' :'8482280198','emergency' :'9794223812'}}
patient1 = Patient(**patient_info)   #object  #validation --> type coercion

In [36]:
insert_patient_data(patient1)

ADITIGUPTA
aditi.gup30@icici.com
http://www.linkedin.com/
65
75.3
['pollen', 'dust', 'abc', 'xyz']
None
BMI 25.45
inserted
